## Домашнее задание к семинару "введение в нейронные сети"

# 1. Затухание градиентов

В домашней работе чуть больше внимания уделим градиентам.     
Для начала посмотрим на графики функций активации и их производных

Еще мы сегодня нарисуем нашу сеть, для этого поставим torchview

In [0]:
%pip install plotly torch torchvision

In [0]:
%pip install torchview -q

In [0]:
import numpy as np
import plotly.express as px
import pandas as pd

In [0]:
def sigmoid(g):
  return 1 / (1 + np.exp(-g))

def der_sigmoid(g):
  a = sigmoid(g)
  return np.multiply(a, 1 - a)

In [0]:
df = pd.DataFrame()
df['x'] = np.linspace(-5, 5, 10_000)
df['сигмоида'] = sigmoid(df['x'])
df['ее_производная'] = der_sigmoid(df['x'])

fig = px.line(df, x='x', y=df.columns[1::])
fig.update_layout(
                  xaxis_title=""
                  , yaxis_title=""
                  , title="Сигмоида и ее производная"
                  , height=600
                  , width=1200
                  , font_family="Arial"
                  , font_color="black"
                  , font_size = 16
                  , title_font_family="Times New Roman"
                  , title_font_size = 18
                  , title_font_color="black"
                  , separators=", .*"
)
fig.show()

In [0]:
def tanh(x):
    return np.tanh(x)

def der_tanh(x):
  a = np.tanh(x)
  return 1 - np.multiply(a, a)

In [0]:
df = pd.DataFrame()
df['x'] = np.linspace(-5, 5, 10_000)
df['tanh'] = tanh(df['x'])
df['производная_tanh'] = der_tanh(df['x'])

fig = px.line(df, x='x', y=df.columns[1::])
fig.update_layout(
                  xaxis_title=""
                  , yaxis_title=""
                  , title="Tanh и его производная"
                  , height=600
                  , width=1200
                  , font_family="Arial"
                  , font_color="black"
                  , font_size = 16
                  , title_font_family="Times New Roman"
                  , title_font_size = 18
                  , title_font_color="black"
                  , separators=", .*"
)
fig.show()

На обоих графиках есть участки где производная околонулевая -- раз мы делаем шаг по граиденту, то попав в такую область, далеко не уйдем.   
Давайте проверим это, возьмем все с семинара

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import IterableDataset
from torch.utils.data.dataloader import DataLoader

from matplotlib import pyplot as plt

import math
torch.backends.cudnn.enabled = True
torch.manual_seed(45)
torch.cuda.manual_seed(45)
torch.use_deterministic_algorithms(mode=True)

In [0]:
from sklearn.datasets import make_classification
import numpy as np

n_features = 30
n_samples = 200_000
data = make_classification(n_samples = n_samples, n_features =n_features, random_state = 42, n_redundant = 10)

In [0]:
def train_test_split(data, test_size=0.33, seed=903):
  rng = np.random.default_rng(seed=seed)
  idx = rng.choice([True, False], size=data[0].shape[0], p=[1 - test_size, test_size])
  return (data[0][idx, :], data[1][idx]), (data[0][~idx, :], data[1][~idx])

train, test = train_test_split(data)

In [0]:
def make_batch(dataset, batch_size = 512):
  data_batched = []
  for k in range(0, dataset[0].shape[0] // batch_size + 1):
    idx_left, idx_right = k * batch_size, min((k + 1) * batch_size, dataset[0].shape[0])
    data_batched.append((dataset[0][idx_left:idx_right, :], dataset[1][idx_left:idx_right]))
  assert (len(data_batched) - 1) * batch_size + data_batched[-1][0].shape[0] == dataset[0].shape[0]
  return data_batched

train_batched = make_batch(train)
test_batched = make_batch(test)

Теперь попробуем воспроизвести эффект затухания градиентов

In [0]:
# очень плохой код!
class SevenLayersFCNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2, hidden_size = nn.Linear(hidden_size, hidden_size - 1), hidden_size - 1
        self.l3, hidden_size = nn.Linear(hidden_size, hidden_size - 1), hidden_size - 1
        self.l4, hidden_size = nn.Linear(hidden_size, hidden_size - 1), hidden_size - 1
        self.l5, hidden_size = nn.Linear(hidden_size, hidden_size - 1), hidden_size - 1
        self.l6, hidden_size = nn.Linear(hidden_size, hidden_size - 1), hidden_size - 1
        self.l7, hidden_size = nn.Linear(hidden_size, hidden_size - 1), hidden_size - 1
        self.lout = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.sigmoid(self.l1(x))
        x = F.sigmoid(self.l2(x))
        x = F.sigmoid(self.l3(x))
        x = F.sigmoid(self.l4(x))
        x = F.sigmoid(self.l5(x))
        x = F.sigmoid(self.l6(x))
        x = F.sigmoid(self.l7(x))
        x = F.sigmoid(self.lout(x))
        return x

In [0]:
# код получше
class DeepFCNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(DeepFCNet, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_size, hidden_size))
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size - 1))
            hidden_size -= 1
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.sigmoid(layer(x))
        return F.sigmoid(self.output_layer(x))

In [0]:
fc_network = DeepFCNet(n_features, 15, 1, num_layers = 10)
criterion = nn.BCELoss()
optimizer = optim.SGD(fc_network.parameters(), lr = 0.1, momentum = 0.01)

Давайте посмотрим на нормы градиентов после обучения на одном батче

In [0]:
for idx, batch in enumerate(train_batched):
  x, y_true = torch.Tensor(batch[0]), torch.Tensor(batch[1].reshape(-1,1))
  optimizer.zero_grad()
  y_pred = fc_network(x)
  loss = criterion(y_pred, y_true)
  loss.backward()
  optimizer.step()
  for name, param in fc_network.named_parameters():
      if (param.grad is not None) and name.find('weight') > -1:
          gradient_norm = np.linalg.norm(param.grad, ord = 2)
          print(f'Gradient of {name}: {gradient_norm}')
  break

Меньше всего меняется слой, наиболее далекий от выхода сети (мы же обучаемся через backpropagation).     
И вообще -- в нашей архитектуре чем дальше слой от выхода тем меньше градиенты

Давайте кстати посмотрим на это "дальше":

In [0]:
svg = graph.visual_graph.pipe(format="svg").decode("utf-8")
displayHTML(svg)

In [0]:
from torchview import draw_graph
import graphviz

graph = draw_graph(fc_network,
                   input_data=(x),
                   expand_nested=True)
graph.visual_graph

Существует много приемов и их сочетаний для починки затухания градиентов :



* смена функции активации на ReLU и ее модификации
- Batch Normalization на каждом слое
- Residual Connections см. ResNet
- Изменение LR
- Gradient Clipping -- ограничить значение градиентов снизу (а если столкнулись с взрывом градиентов -- сверху)


Попробуем tanh в качестве функции активации

In [0]:
class DeepFCNetTanh(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(DeepFCNetTanh, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_size, hidden_size))
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size - 1))
            hidden_size -= 1
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.tanh(layer(x))
        return F.sigmoid(self.output_layer(x))

In [0]:
fc_network = DeepFCNetTanh(n_features, 15, 1, num_layers = 10)
criterion = nn.BCELoss()
optimizer = optim.SGD(fc_network.parameters(), lr = 0.1, momentum = 0.01)

for idx, batch in enumerate(train_batched):
  x, y_true = torch.Tensor(batch[0]), torch.Tensor(batch[1].reshape(-1,1))
  optimizer.zero_grad()
  y_pred = fc_network(x)
  loss = criterion(y_pred, y_true)
  loss.backward()
  optimizer.step()
  for name, param in fc_network.named_parameters():
      if (param.grad is not None) and name.find('weight') > -1:
        try:
          gradient_norm = np.linalg.norm(param.grad, ord = 2)
          print(f'Gradient of {name}: {gradient_norm}')
        except Exception as ex:
          print(ex)
          print(f'Gradient of {name}: {param.grad}')
      else:
          pass
  break

В теории у tanh проблема такая же, но ситуация заметно лучше

попробуем ReLU

In [0]:
class DeepFCNetRelu(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_size, hidden_size))
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size - 1))
            hidden_size -= 1
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return F.sigmoid(self.output_layer(x))

In [0]:
fc_network = DeepFCNetRelu(n_features, 15, 1, num_layers = 10)
criterion = nn.BCELoss()
optimizer = optim.SGD(fc_network.parameters(), lr = 0.1, momentum = 0.01)

for idx, batch in enumerate(train_batched):
  x, y_true = torch.Tensor(batch[0]), torch.Tensor(batch[1].reshape(-1,1))
  optimizer.zero_grad()
  y_pred = fc_network(x)
  loss = criterion(y_pred, y_true)
  loss.backward()
  optimizer.step()
  for name, param in fc_network.named_parameters():
      if (param.grad is not None) and name.find('weight') > -1:
        try:
          gradient_norm = np.linalg.norm(param.grad, ord = 2)
          print(f'Gradient of {name}: {gradient_norm}')
        except Exception as ex:
          print(ex)
          print(f'Gradient of {name}: {param.grad}')
      else:
          pass
  break

Не помогло!

## Домашнее задание:

попробовать 4 указанных ниже приема для борьбы с затуханием градиента на этом же датасете с архитектурой как в классе DeepFCNet и сказать какой прием сработал лучше

*   LeakyReLU (для самых дотошных -- [Mish](https://krutikabapat.github.io/Swish-Vs-Mish-Latest-Activation-Functions/))
*   Batch Normalization на каждом слое
*   Gradient Clipping (снизу)
*   менять LR в процессе обучения


# LeakyReLU

In [0]:
class DeepFCNetRelu(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_size, hidden_size))
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size - 1))
            hidden_size -= 1
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.leaky_relu(layer(x))
        return F.sigmoid(self.output_layer(x))
    
fc_network = DeepFCNetRelu(n_features, 15, 1, num_layers = 10)
criterion = nn.BCELoss()
optimizer = optim.SGD(fc_network.parameters(), lr = 0.1, momentum=0.01)

for idx, batch in enumerate(train_batched):
  x, y_true = torch.Tensor(batch[0]), torch.Tensor(batch[1].reshape(-1,1))
  optimizer.zero_grad()
  y_pred = fc_network(x)
  loss = criterion(y_pred, y_true)
  loss.backward()
  optimizer.step()
  for name, param in fc_network.named_parameters():
      if (param.grad is not None) and name.find('weight') > -1:
        try:
          gradient_norm = np.linalg.norm(param.grad, ord = 2)
          print(f'Gradient of {name}: {gradient_norm}')
        except Exception as ex:
          print(ex)
          print(f'Gradient of {name}: {param.grad}')
      else:
          pass
  break

# Batch Norm + Relu + Adam

In [0]:
class DeepFCNetRelu(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(
            nn.Sequential(
                nn.Linear(input_size, hidden_size),
                nn.BatchNorm1d(hidden_size),
            )
        )
        for _ in range(num_layers - 1):
            self.layers.append(
                nn.Sequential(
                    nn.Linear(hidden_size, hidden_size - 1),
                    nn.BatchNorm1d(hidden_size - 1),
                )
            )
            hidden_size -= 1
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return F.sigmoid(self.output_layer(x))


fc_network = DeepFCNetRelu(n_features, 15, 1, num_layers=10)
criterion = nn.BCELoss()
# optimizer = optim.SGD(fc_network.parameters(), lr=0.1, momentum=0.01)
optimizer = optim.Adam(fc_network.parameters(), lr=0.1)

for idx, batch in enumerate(train_batched):
    x, y_true = torch.Tensor(batch[0]), torch.Tensor(batch[1].reshape(-1, 1))
    optimizer.zero_grad()
    y_pred = fc_network(x)
    loss = criterion(y_pred, y_true)
    loss.backward()
    optimizer.step()
    for name, param in fc_network.named_parameters():
        if (param.grad is not None) and name.find("weight") > -1:
            try:
                gradient_norm = np.linalg.norm(param.grad, ord=2)
                print(f"Gradient of {name}: {gradient_norm}")
            except Exception as ex:
                print(ex)
                print(f"Gradient of {name}: {param.grad}")
        else:
            pass
    break

# LR Chaning

In [0]:
class DeepFCNetRelu(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_size, hidden_size))
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size - 1))
            hidden_size -= 1
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.leaky_relu(layer(x))
        return F.sigmoid(self.output_layer(x))
    
fc_network = DeepFCNetRelu(n_features, 15, 1, num_layers = 10)
criterion = nn.BCELoss()
optimizer = optim.SGD(fc_network.parameters(), lr = 0.1, momentum=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

for idx, batch in enumerate(train_batched):
  x, y_true = torch.Tensor(batch[0]), torch.Tensor(batch[1].reshape(-1,1))
  optimizer.zero_grad()
  y_pred = fc_network(x)
  loss = criterion(y_pred, y_true)
  loss.backward()
  optimizer.step()
  scheduler.step()
  for name, param in fc_network.named_parameters():
      if (param.grad is not None) and name.find('weight') > -1:
        try:
          gradient_norm = np.linalg.norm(param.grad, ord = 2)
          print(f'Gradient of {name}: {gradient_norm}')
        except Exception as ex:
          print(ex)
          print(f'Gradient of {name}: {param.grad}')
      else:
          pass
  break